# Text Embedding using Word2Vec


In [ ]:
!pip install gensim umap-learn -q

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
save_path = "/content/drive/MyDrive/embedding/"
os.makedirs(save_path, exist_ok=True)
print(f"Save path: {save_path} ")

In [ ]:
import pandas as pd
import numpy as np
from gensim.models import Word2Vec
import time
df = pd.read_csv(f"{save_path}cleaned_sentiment_dataset.csv")
print(f"Dataset shape: {df.shape}")
print(df[['clean_text', 'label']].head())

In [ ]:
df = df.dropna(subset=['clean_text'])
df['clean_text'] = df['clean_text'].astype(str)
print(f"Rows after cleaning: {len(df)}")

In [ ]:

tokenized_texts = [text.split() for text in df['clean_text']]
print(f"Sample tokens: {tokenized_texts[0][:10]}")
print(f"Total sentences: {len(tokenized_texts)}")

In [ ]:
print("Training Word2Vec model...")
start = time.time()

w2v_model = Word2Vec(
    sentences=tokenized_texts,
    vector_size=300,   
    window=5,          
    min_count=2,       
    workers=4,         
    epochs=10,         
    sg=1               
    )

elapsed = time.time() - start
print(f"Training done in {elapsed:.1f}s ")
print(f"Vocabulary size: {len(w2v_model.wv)}")

In [ ]:
test_word = "depression"
if test_word in w2v_model.wv:
    similar = w2v_model.wv.most_similar(test_word, topn=5)
    print(f"Most similar words to '{test_word}':")
    for word, score in similar:
        print(f"  {word}: {score:.3f}")
else:
    print(f"'{test_word}' not in vocabulary, try another word")

In [ ]:

def get_sentence_embedding(tokens, model, vector_size=300):
    vectors = [
        model.wv[word]
        for word in tokens
        if word in model.wv
    ]
    if vectors:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(vector_size) 

print("Generating sentence embeddings...")
start = time.time()

embeddings = np.array([
    get_sentence_embedding(tokens, w2v_model)
    for tokens in tokenized_texts
])

elapsed = time.time() - start
print(f"Done in {elapsed:.1f}s ")
print(f"Embeddings shape: {embeddings.shape}")  

In [ ]:
print("Sample text:", df['clean_text'].iloc[0])
print("Embedding vector (first 10 dims):", embeddings[0][:10])
print("Embedding dimension:", embeddings.shape[1])

zero_rows = np.all(embeddings == 0, axis=1).sum()
print(f"Zero embedding rows: {zero_rows} ({zero_rows/len(embeddings)*100:.1f}%)")

In [ ]:

w2v_model.save(f"{save_path}word2vec_model.model")
print("word2vec_model.model saved")

np.save(f"{save_path}word2vec_embeddings.npy", embeddings)
print("word2vec_embeddings.npy saved ")

df[['label']].to_csv(f"{save_path}embedding_labels.csv", index=False)
print("embedding_labels.csv saved ")

embedding_df = pd.DataFrame(embeddings, columns=[f'emb_{i}' for i in range(embeddings.shape[1])])
final_df = pd.concat([df[['clean_text', 'label']].reset_index(drop=True), embedding_df], axis=1)
final_df.to_csv(f"{save_path}dataset_with_embeddings.csv", index=False)
print(f"dataset_with_embeddings.csv saved ✅  Shape: {final_df.shape}")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

sample_emb = embeddings[:3]
sim_matrix = cosine_similarity(sample_emb)

print("Cosine Similarity Matrix:")
print(np.round(sim_matrix, 3))
for i in range(3):
    print(f"\n[{i}] {df['clean_text'].iloc[i][:80]}...")

In [ ]:
import umap
import matplotlib.pyplot as plt

sample_size = min(1000, len(embeddings))
sample_idx = np.random.choice(len(embeddings), sample_size, replace=False)
sample_emb = embeddings[sample_idx]
sample_labels = df['label'].iloc[sample_idx].values

reducer = umap.UMAP(n_components=2, random_state=42)
reduced = reducer.fit_transform(sample_emb)

plt.figure(figsize=(10, 7))
for label in np.unique(sample_labels):
    mask = sample_labels == label
    plt.scatter(reduced[mask, 0], reduced[mask, 1], label=label, alpha=0.6, s=10)

plt.title("Word2Vec Embeddings - UMAP Visualization")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig(f"{save_path}word2vec_umap.png", dpi=150)
plt.show()
print("UMAP visualization saved to Drive ✅")

In [ ]:
files = os.listdir(save_path)
print(f"Files saved in {save_path}:")
for f in sorted(files):
    size = os.path.getsize(os.path.join(save_path, f)) / (1024 * 1024)
    print(f"  {f}  ({size:.1f} MB)")

## الملفات المحفوظة على الدرايف

| الملف | الوصف |
|---|---|
| `word2vec_model.model` | الموديل كامل (تقدر تدرب عليه أو تستخدمه تاني) |
| `word2vec_embeddings.npy` | الـ embeddings كـ numpy array `(n_samples, 300)` |
| `embedding_labels.csv` | الـ labels المقابلة لكل صف |
| `dataset_with_embeddings.csv` | `clean_text` + `label` + 300 embedding columns |
| `word2vec_umap.png` | تصور الـ embeddings في 2D |

## الفرق بين Word2Vec و SBERT

| | Word2Vec | SBERT |
|---|---|---|
| **الطريقة** | متوسط vectors الكلمات | موديل كامل للجملة |
| **السياق** |  مش بيفهم السياق |  بيفهم الجملة كوحدة |
| **الحجم** | 300 dim | 384 dim |
| **السرعة** | أسرع في التدريب | أبطأ لكن أدق |
| **الأفضل لـ** | كلمات متشابهة | جمل متشابهة |
